In [22]:
!pip install transformers datasets torch accelerate


In [23]:
import transformers
import datasets
import torch

print("Libraries installed successfully!")

Libraries installed successfully!


In [24]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("GPT-2 loaded successfully!")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 loaded successfully!


In [25]:
text = """
Artificial Intelligence is transforming industries worldwide.
Machine Learning enables systems to learn from data.
Deep Learning is a powerful subset of Machine Learning.
Natural Language Processing helps computers understand human language.
Data Science combines statistics and programming.
"""

with open("data.txt", "w") as f:
    f.write(text)

print("Dataset created successfully!")

Dataset created successfully!


In [26]:
from datasets import load_dataset

dataset = load_dataset("text", data_files={"train": "data.txt"})

print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 6
    })
})


In [27]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenizer.pad_token = tokenizer.eos_token
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Tokenization completed!")

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenization completed!


In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10
)

print("Training setup completed!")

Training setup completed!


In [29]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.331647


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18, training_loss=2.6761912769741483, metrics={'train_runtime': 124.6768, 'train_samples_per_second': 0.144, 'train_steps_per_second': 0.144, 'total_flos': 1175814144000.0, 'train_loss': 2.6761912769741483, 'epoch': 3.0})

In [21]:
prompt = "Artificial Intelligence"

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    **inputs,
    max_length=60,
    do_sample=True,
    temperature=0.8,
    num_return_sequences=1
)

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print(generated_text)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Artificial Intelligence (AI) is advancing artificial intelligence technology. With AI is transforming everything from human to machine. With AI can transform complex business logic, biology, and even biology, from information processing to computer science and technology.


As of April 2018, AI is now the number 2 ranked technology
